# Семинар: Pandas

Этот практикум основан на материалах лекции по Pandas (Series/DataFrame, индексация, маскирование, выравнивание, NaN, агрегирование, merge, groupby).  
Перед началом **обязательно** задайте `STUDENT_ID` в ячейке ниже: от него зависит генерация данных и ответы в проверках.

## Правила
- Разрешены: `pandas`, `numpy` (и стандартная библиотека Python).
- Запрещены: ручной перебор строк `for row in df.itertuples()` / `iterrows()` и т.п. там, где можно сделать векторно (в некоторых заданиях это проверяется по времени).
- Не меняйте код в ячейках **«Тесты (не изменять)»**.

## Структура
- Пара 1: задания 01–10
- Пара 2: задания 11–20


### Подготовка окружения и генерация данных

In [ ]:

# Базовые импорты
import hashlib
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# -----------------------------
# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"

def _seed_from_tag(tag: str, salt: str = "pandas-seminar-2026") -> int:
    if not isinstance(tag, str) or len(tag.strip()) != 6:
        raise ValueError("STUDENT_ID должен быть строкой длиной == 6 символов.")
    h = hashlib.sha256((salt + "|" + tag.strip()).encode("utf-8")).hexdigest()
    return int(h[:8], 16)  # 32-bit

def make_data(tag: str):
    rs = np.random.RandomState(_seed_from_tag(tag))

    # ---------- 1) «states» как в лекции: population + area -> density ----------
    states_list = ["California","Texas","New York","Florida","Illinois","Pennsylvania","Ohio","Georgia"]
    pop = pd.Series(rs.randint(8_000_000, 40_000_000, size=len(states_list)), index=states_list, name="population")
    area = pd.Series(rs.randint(40_000, 720_000, size=len(states_list)), index=states_list, name="area")
    states = pd.DataFrame({"population": pop, "area": area})
    states["density"] = states["population"] / states["area"]

    # ---------- 2) Данные для выравнивания (alignment) ----------
    idx1 = list(range(0, 5))
    idx2 = list(range(2, 8))
    cols1 = list("ABCD")
    cols2 = list("BCDF")
    df_a = pd.DataFrame(rs.randint(0, 10, (len(idx1), len(cols1))), index=idx1, columns=cols1)
    df_b = pd.DataFrame(rs.randint(0, 10, (len(idx2), len(cols2))), index=idx2, columns=cols2)

    # ---------- 3) HR-таблицы для merge (1-1, m-1, m-m) ----------
    employees = pd.DataFrame({
        "employee": ["Bob","Jake","Lisa","Sue","Mary","John","Anya","Oleg"],
        "group": rs.choice(["Accounting","Engineering","HR","Sales"], size=8, replace=True)
    })
    hire = pd.DataFrame({
        "employee": employees["employee"].sample(frac=1.0, random_state=rs).values,
        "hire_date": rs.randint(2000, 2025, size=8)
    })
    supervisors = pd.DataFrame({
        "group": ["Accounting","Engineering","HR","Sales"],
        "supervisor": ["Carly","Guido","Steve","Olivia"]
    })
    # навыки: многие-ко-многим по group
    skill_pool = {
        "Accounting": ["math","spreadsheets","tax"],
        "Engineering": ["coding","linux","ml"],
        "HR": ["organization","spreadsheets","interviews"],
        "Sales": ["negotiation","crm","presentation"],
    }
    rows = []
    for g, skills in skill_pool.items():
        for s in skills:
            if rs.rand() < 0.85:  # немного «пропусков» навыков
                rows.append({"group": g, "skills": s})
    skills = pd.DataFrame(rows)

    # ---------- 4) «planets» (похожее на пример из лекции) ----------
    methods = [
        "Radial Velocity","Transit","Imaging","Microlensing",
        "Astrometry","Pulsar Timing","Eclipse Timing Variations"
    ]
    n = 900
    planets = pd.DataFrame({
        "method": rs.choice(methods, size=n, p=np.array([0.45,0.35,0.05,0.06,0.02,0.03,0.04])),
        "number": rs.randint(1, 4, size=n),
        "orbital_period": np.round(np.exp(rs.normal(4.2, 1.2, size=n)), 3),  # логнормальное распределение
        "mass": np.round(np.exp(rs.normal(0.2, 1.0, size=n)), 3),
        "distance": np.round(np.abs(rs.normal(80, 60, size=n)), 2),
        "year": rs.randint(1990, 2016, size=n)
    })
    # добавим пропуски
    for col, p_nan in [("orbital_period", 0.03), ("mass", 0.35), ("distance", 0.12)]:
        mask = rs.rand(n) < p_nan
        planets.loc[mask, col] = np.nan

    # ---------- 5) «retail» транзакции для end-to-end пайплайна ----------
    n_products = 18
    categories = ["A","B","C","D"]
    products = pd.DataFrame({
        "product_id": [f"P{str(i).zfill(3)}" for i in range(n_products)],
        "category": rs.choice(categories, size=n_products),
        "cost": np.round(rs.uniform(5.0, 70.0, size=n_products), 2),
    })
    # «рекомендованная» цена с шумом
    products["base_price"] = np.round(products["cost"] * rs.uniform(1.25, 2.2, size=n_products), 2)

    n_customers = 120
    customers = pd.DataFrame({
        "customer_id": [f"C{str(i).zfill(4)}" for i in range(n_customers)],
        "region": rs.choice(["North","South","West","East"], size=n_customers),
        "segment": rs.choice(["new","regular","vip"], size=n_customers, p=[0.35,0.55,0.10]),
        "signup_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(rs.randint(0, 500, size=n_customers), unit="D"),
    })

    n_tx = 6000
    dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(rs.randint(0, 365, size=n_tx), unit="D")
    transactions = pd.DataFrame({
        "tx_id": [f"T{str(i).zfill(6)}" for i in range(n_tx)],
        "date": dates,
        "store": rs.choice(["S1","S2","S3","S4","S5"], size=n_tx),
        "product_id": rs.choice(products["product_id"].values, size=n_tx),
        "customer_id": rs.choice(customers["customer_id"].values, size=n_tx),
        "units": rs.poisson(2.2, size=n_tx) + 1,
        "promo": rs.choice([0, 1, None], size=n_tx, p=[0.75, 0.20, 0.05]),
    })

    # Цена транзакции = base_price + шум; часть цен — NaN
    tx_price = products.set_index("product_id")["base_price"].reindex(transactions["product_id"]).values
    tx_price = np.asarray(tx_price, dtype=float) * rs.uniform(0.90, 1.10, size=n_tx)
    tx_price = np.round(tx_price, 2)
    nan_mask = rs.rand(n_tx) < 0.06
    tx_price[nan_mask] = np.nan
    transactions["price"] = tx_price

    # Сделаем небольшую долю отрицательных/нулевых units как «грязные данные»
    bad_units = rs.rand(n_tx) < 0.01
    transactions.loc[bad_units, "units"] = rs.choice([0, -1], size=bad_units.sum())

    return {
        "states": states,
        "df_a": df_a,
        "df_b": df_b,
        "employees": employees,
        "hire": hire,
        "supervisors": supervisors,
        "skills": skills,
        "planets": planets,
        "products": products,
        "customers": customers,
        "transactions": transactions,
    }

DATA = make_data(STUDENT_ID)

# Удобные короткие ссылки (используй в заданиях)
states = DATA["states"]
df_a = DATA["df_a"]
df_b = DATA["df_b"]
employees = DATA["employees"]
hire = DATA["hire"]
supervisors = DATA["supervisors"]
skills = DATA["skills"]
planets = DATA["planets"]
products = DATA["products"]
customers = DATA["customers"]
transactions = DATA["transactions"]

print("DATA готов. Пример таблиц:")
display(states.head())
display(df_a.head())
display(planets.head())
display(transactions.head())


DATA готов. Пример таблиц:


,population,area,density
California,25021317,264889,94.459630
Texas,28880744,553809,52.149286
New York,32974137,85926,383.750402
Florida,39159816,481121,81.392864
Illinois,26842531,162288,165.400590


,A,B,C,D
0,1,1,8,1
1,8,3,4,7
2,0,5,4,3
3,2,2,7,5
4,9,5,0,9


,method,number,orbital_period,mass,distance,year
0,Radial Velocity,3,45.774,3.100,57.98,1991
1,Imaging,3,42.117,0.347,NaN,1992
2,Radial Velocity,1,47.233,2.569,8.65,1991
3,Transit,3,17.436,0.417,81.17,2010
4,Pulsar Timing,3,146.481,4.084,77.53,1993


,tx_id,date,store,product_id,customer_id,units,promo,price
0,T000000,2024-11-20,S5,P008,C0106,5,0,72.65
1,T000001,2024-06-18,S2,P016,C0013,3,0,NaN
2,T000002,2024-12-07,S3,P000,C0011,3,0,104.88
3,T000003,2024-11-04,S1,P009,C0047,5,1,127.95
4,T000004,2024-01-26,S3,P008,C0104,1,0,84.85


## Пара 1 (01–10): Series, DataFrame, индексация, маскирование, alignment, NaN, агрегирование

### Series: явная переиндексация из словаря (reindex)

**Задание 01.** Напиши функцию `task01_series_reindex(mapping, index_order)`, которая:
- создаёт `pd.Series` из `mapping` (обычный `dict`);
- затем *явно* переиндексирует её в порядке `index_order` (включая ключи, которых нет в `mapping` → должны стать `NaN`);
- задаёт `name="reports"`.

Ограничение: решение должно быть векторизованным (без циклов).

In [ ]:
# YOUR CODE HERE
def task01_series_reindex(mapping: dict, index_order: list) -> pd.Series:
    """Возвращает Series с индексом index_order и name='reports'."""
    raise NotImplementedError()


In [ ]:
# Тесты к заданию 01 (не изменять)
m = {"Cochice": 4, "Pima": 24, "Yuma": 3}
order = ["Cochice","Pima","Santa Cruz","Maricopa","Yuma"]
out = task01_series_reindex(m, order)
assert isinstance(out, pd.Series)
assert list(out.index) == order
assert out.name == "reports"
assert out.isna().sum() == 2
assert out.loc["Pima"] == 24

### Series: обновление значений без изменения исходного объекта

**Задание 02.** Напиши `task02_series_apply_updates(sr, updates)`, которая:
- принимает `sr: pd.Series` и `updates: dict`;
- возвращает **новую** серию, где для каждого ключа из `updates` значение обновлено/добавлено;
- исходная серия `sr` не должна измениться.

Подсказка: в лекции показано добавление/изменение элемента Series через индекс (см. раздел про индексацию Series).

In [ ]:
def task02_series_apply_updates(sr: pd.Series, updates: dict) -> pd.Series:
    """Возвращает новую Series с применёнными updates, не модифицируя sr."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 02 (не изменять)
sr0 = pd.Series([0.25, 0.5, 0.75, 1.0], index=list("abcd"))
sr1 = task02_series_apply_updates(sr0, {"e": 1.25, "b": 0.55})
assert "e" in sr1.index and sr1.loc["e"] == 1.25
assert sr1.loc["b"] == 0.55
assert "e" not in sr0.index, "Исходная серия должна остаться без 'e'"
assert sr0.loc["b"] == 0.5, "Исходная серия не должна измениться"

### Series: срез по меткам с включением правой границы (loc)

**Задание 03.** Напиши `task03_slice_inclusive(sr, start, end)`, которая:
- делает срез Series по **меткам** (label-based) так, чтобы **правая граница включалась** в результат;
- возвращает срез как `pd.Series`.

Ожидается использование `.loc[...]` (в лекции подчёркнуто, что при явных индексах правая граница среза включается).

In [ ]:
def task03_slice_inclusive(sr: pd.Series, start, end) -> pd.Series:
    """Срез по меткам [start:end] с включением end."""
    # YOUR CODE HERE

In [ ]:

# Тесты к заданию 03 (не изменять)
sr = pd.Series([10, 20, 30, 40, 50], index=["a","b","c","d","e"])
out = task03_slice_inclusive(sr, "b", "d")
assert list(out.index) == ["b","c","d"]
assert out.loc["d"] == 40

### Series: прихотливое индексирование с сохранением порядка и повторов

**Задание 04.** Напиши `task04_fancy_select(sr, keys)`, которая:
- возвращает элементы `sr` в порядке `keys`;
- если в `keys` есть повторы, они должны повториться и в результате.

Пример поведения соответствует `sr.loc[keys]`. Циклы не нужны.

In [ ]:
def task04_fancy_select(sr: pd.Series, keys: list) -> pd.Series:
    """Возвращает sr.loc[keys] (с сохранением порядка и повторов)."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 04 (не изменять)
sr = pd.Series([1.0, 2.0, 3.0], index=["x","y","z"])
out = task04_fancy_select(sr, ["y","x","y"])
assert list(out.index) == ["y","x","y"]
assert out.iloc[0] == 2.0 and out.iloc[2] == 2.0

### Series: булевое маскирование по диапазону + тест на векторизацию

**Задание 05.** Напиши `task05_mask_between(sr, lo, hi, inclusive=True)`, которая:
- возвращает элементы серии, попадающие в диапазон `(lo, hi)` или `[lo, hi]` (в зависимости от `inclusive`);
- использует булевы маски и логические операции `&` (как в лекции для маскирования Series).

**Важно:** решение должно быть векторизованным (без циклов). В тестах есть проверка на скорость.

In [ ]:
def task05_mask_between(sr: pd.Series, lo: float, hi: float, inclusive: bool = True) -> pd.Series:
    """Фильтрует Series по диапазону, используя булевы маски."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 05 (не изменять)
import time

rs = np.random.RandomState(0)
sr = pd.Series(rs.randn(300_000))
t0 = time.perf_counter()
out = task05_mask_between(sr, -0.2, 0.2, inclusive=True)
dt = time.perf_counter() - t0

assert isinstance(out, pd.Series)
assert out.min() >= -0.2 - 1e-12
assert out.max() <= 0.2 + 1e-12
assert dt < 1.0, f"Слишком медленно ({dt:.3f}s). Скорее всего, использован цикл."

### DataFrame: построение из двух словарей с выравниванием и плотностью

**Задание 06.** Напиши `task06_build_states(population, area)`, которая:
- принимает два словаря `population` и `area`, где ключи — названия штатов, значения — числа;
- возвращает `DataFrame` с колонками `population`, `area`, `density`;
- индексы должны быть **объединением** ключей двух словарей, отсортированным по алфавиту;
- `density = population / area` (если чего-то не хватает → `NaN`).

Циклы не нужны: используй выравнивание индексов в Pandas (см. раздел про DataFrame как словарь Series и выравнивание).

In [ ]:
def task06_build_states(population: dict, area: dict) -> pd.DataFrame:
    """Строит DataFrame population/area/density с выравниванием индексов."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 06 (не изменять)
pop = {"B": 10, "A": 20}
area = {"A": 5, "C": 2}
df = task06_build_states(pop, area)
assert list(df.columns) == ["population","area","density"]
assert list(df.index) == ["A","B","C"]
assert np.isclose(df.loc["A","density"], 20/5)
assert pd.isna(df.loc["B","area"]) and pd.isna(df.loc["B","density"])

### DataFrame: фильтрация строк по условию и выбор столбцов через loc

**Задание 07.** Напиши `task07_states_high_density(df, q=0.75)`, которая:
- вычисляет порог `thr = df['density'].quantile(q)` (квантиль);
- выбирает строки, где `density > thr`;
- возвращает только колонки `population` и `density`;
- сортирует результат по `density` по убыванию.

Ожидается использование `loc` (см. лекцию: `states.loc[mask, ['population','density']]`).

In [ ]:
def task07_states_high_density(df: pd.DataFrame, q: float = 0.75) -> pd.DataFrame:
    """Фильтрует по квантилю плотности и возвращает population/density, сортируя по density desc."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 07 (не изменять)
df = states.copy()
out = task07_states_high_density(df, q=0.6)
assert list(out.columns) == ["population","density"]
thr = df["density"].quantile(0.6)
assert (out["density"] > thr).all()
assert out["density"].is_monotonic_decreasing

### Выравнивание (alignment): сложение DataFrame и отчёт по числу NaN

**Задание 08.** Напиши `task08_alignment_add(df1, df2)`, которая возвращает кортеж `(df_sum, nan_count)`:
- `df_sum` — результат `df1 + df2` (важно: именно обычное сложение с автоматическим выравниванием);
- `nan_count` — количество `NaN` в `df_sum` (суммарно по всем ячейкам).

Проверь себя на данных `df_a` и `df_b` из `DATA`.

In [ ]:
def task08_alignment_add(df1: pd.DataFrame, df2: pd.DataFrame) -> tuple:
    """Складывает df1 и df2 с выравниванием и возвращает (сумма, число NaN)."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 08 (не изменять)
df_sum, nan_count = task08_alignment_add(df_a, df_b)
assert isinstance(df_sum, pd.DataFrame)
assert isinstance(nan_count, (int, np.integer))
assert nan_count == int(df_sum.isna().sum().sum())
# Должна быть хотя бы одна NaN из-за несовпадения индексов/столбцов
assert nan_count > 0

### NaN: dropna(how='all') -> dropna(axis=1) -> fillna(0.0)

**Задание 09.** Напиши `task09_dropna_chain(df)`, которая:
1) удаляет строки, где **все** значения `NaN` (`dropna(how='all')`);
2) затем удаляет столбцы, где есть **хотя бы один** `NaN` (после шага 1);
3) затем заполняет оставшиеся `NaN` значением `0.0`.

Функция должна вернуть **новый** DataFrame и не менять исходный `df` (без `inplace=True`).

In [ ]:
def task09_dropna_chain(df: pd.DataFrame) -> pd.DataFrame:
    """Выполняет цепочку dropna/dropna/fillna и возвращает новый DataFrame."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 09 (не изменять)
df = pd.DataFrame({"A":[np.nan, 1.0, 2.0, np.nan],
                   "B":[np.nan, 3.0, np.nan, np.nan],
                   "C":[np.nan, 4.0, 5.0, np.nan]})
df_copy = df.copy(deep=True)

out = task09_dropna_chain(df)

assert df.equals(df_copy), "Исходный df не должен изменяться."
assert isinstance(out, pd.DataFrame)
assert out.isna().sum().sum() == 0, "В результате не должно быть NaN"
# После удаления строк all-NaN останутся строки 1 и 2; после удаления столбцов с NaN останется только C и A?
# Но B содержит NaN в строке 2, значит B удаляется. A и C без NaN в строках 1,2.
assert set(out.columns) == {"A","C"}

### Агрегирование: сводная таблица статистик по столбцам

**Задание 10.** Напиши `task10_stats_table(df)`, которая строит таблицу статистик по **числовым** столбцам.

Формат результата:
- индекс: имена исходных столбцов;
- колонки (ровно в таком порядке): `['sum','prod','mean','median','std','var','min','max']`.

Используй методы Pandas (`sum`, `prod`, `mean`, ...). Пропуски должны игнорироваться по стандартным правилам Pandas (`skipna=True`).

In [ ]:
def task10_stats_table(df: pd.DataFrame) -> pd.DataFrame:
    """Возвращает DataFrame статистик по числовым столбцам."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 10 (не изменять)
df = pd.DataFrame({"A":[1,2,3,np.nan], "B":[2,2,2,2]})
out = task10_stats_table(df)
assert list(out.columns) == ["sum","prod","mean","median","std","var","min","max"]
assert list(out.index) == ["A","B"]
assert np.isclose(out.loc["B","std"], 0.0)
assert np.isclose(out.loc["A","sum"], 6.0)

## Пара 2 (11–20): quantile, merge, groupby (count/median/agg/filter/transform), end-to-end пайплайн

### Квантили: таблица quantile для заданных вероятностей

**Задание 11.** Напиши `task11_quantiles(df, probs)`, которая:
- берёт только числовые столбцы;
- вычисляет `quantile` для вероятностей `probs`;
- возвращает DataFrame, где индекс — отсортированный список уникальных `probs`.

Пример: если `probs=[0.5, 0.1, 0.5]`, то в результате индекс должен быть `[0.1, 0.5]`.

In [ ]:
def task11_quantiles(df: pd.DataFrame, probs: list) -> pd.DataFrame:
    """Возвращает таблицу квантилей по числовым столбцам."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 11 (не изменять)
df = pd.DataFrame({"A":[0,1,2,3,4], "B":[10,10,10,10,10], "C":["x","y","z","w","q"]})
out = task11_quantiles(df, [0.5, 0.1, 0.5])
assert list(out.index) == [0.1, 0.5]
assert list(out.columns) == ["A","B"]
assert np.isclose(out.loc[0.5, "A"], 2.0)

### Merge 1-to-1: сотрудники + дата найма

**Задание 12.** Напиши `task12_merge_employee_hire(employees, hire)`, которая:
- объединяет таблицы `employees` и `hire` по столбцу `employee`;
- возвращает DataFrame с колонками `employee, group, hire_date`;
- сортирует строки по `employee` по возрастанию;
- проверка соответствия 1-to-1 выполняется параметром `validate='one_to_one'` (если используешь `pd.merge`).

Используй данные `employees` и `hire` из `DATA`.

In [ ]:
def task12_merge_employee_hire(employees: pd.DataFrame, hire: pd.DataFrame) -> pd.DataFrame:
    """1-to-1 merge employees и hire по employee, сортируя по employee."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 12 (не изменять)
out = task12_merge_employee_hire(employees, hire)
assert list(out.columns) == ["employee","group","hire_date"]
assert out["employee"].is_monotonic_increasing
assert len(out) == len(employees)

### Merge many-to-one: добавление руководителя группы

**Задание 13.** Напиши `task13_add_supervisor(emp, supervisors)`, которая:
- берёт результат задания 12 (таблица с `employee, group, hire_date`);
- добавляет колонку `supervisor`, соединив по `group`;
- гарантирует отсутствие пропусков в `supervisor` (если группа неизвестна — это ошибка данных).

Подсказка: это соединение «многие-к-одному».

In [ ]:
def task13_add_supervisor(emp: pd.DataFrame, supervisors: pd.DataFrame) -> pd.DataFrame:
    """Добавляет supervisor через merge по group."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 13 (не изменять)
emp = task12_merge_employee_hire(employees, hire)
out = task13_add_supervisor(emp, supervisors)
assert "supervisor" in out.columns
assert out["supervisor"].isna().sum() == 0
assert len(out) == len(emp)

### Merge many-to-many: разворачивание навыков сотрудников по группам

**Задание 14.** Напиши `task14_expand_skills(employees, skills)`, которая:
- соединяет `employees` (employee, group) с `skills` (group, skills) по `group`;
- возвращает DataFrame с колонками `employee, group, skills`;
- это соединение «многие-ко-многим», т.к. в каждой группе много сотрудников и много навыков.

Важно: не удаляй дубликаты — если они появляются как результат join, это ожидаемое поведение.

In [ ]:
def task14_expand_skills(employees: pd.DataFrame, skills: pd.DataFrame) -> pd.DataFrame:
    """m-m merge employees и skills по group."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 14 (не изменять)
out = task14_expand_skills(employees, skills)
assert set(out.columns) == {"employee","group","skills"}
assert len(out) >= len(employees)
# Должны быть повторяющиеся employee (у сотрудника несколько навыков группы)
assert out["employee"].duplicated().any()

### GroupBy: count по годам (не-NaN в столбцах)

**Задание 15.** Напиши `task15_year_counts(planets)`, которая:
- группирует `planets` по `year`;
- возвращает таблицу `count()` по столбцам `['method','number','orbital_period','mass','distance']`;
- индекс в результате должен быть отсортирован по возрастанию года.

Это повторяет паттерн из лекции: `planets.groupby('year').count()` (но мы ограничиваем список столбцов).

In [ ]:
def task15_year_counts(planets: pd.DataFrame) -> pd.DataFrame:
    """Группирует по year и считает count по выбранным столбцам."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 15 (не изменять)
out = task15_year_counts(planets)
need_cols = ["method","number","orbital_period","mass","distance"]
assert list(out.columns) == need_cols
assert out.index.is_monotonic_increasing
assert (out[need_cols] >= 0).all().all()

### GroupBy: медиана orbital_period по методу обнаружения

**Задание 16.** Напиши `task16_method_median_orbital(planets)`, которая:
- группирует по `method`;
- считает медиану `orbital_period` (игнорируя NaN);
- возвращает `Series`, отсортированную по убыванию медианы.

In [ ]:
def task16_method_median_orbital(planets: pd.DataFrame) -> pd.Series:
    """Медиана orbital_period по method, сортировка desc."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 16 (не изменять)
out = task16_method_median_orbital(planets)
assert isinstance(out, pd.Series)
assert out.index.is_unique
assert out.is_monotonic_decreasing

### GroupBy.aggregate: min/median/max для orbital_period по method

**Задание 17.** Напиши `task17_method_agg_orbital(planets)`, которая возвращает DataFrame:
- индекс: `method`;
- колонки: `min`, `median`, `max` (ровно в таком порядке) для `orbital_period`.

Разрешено использовать `aggregate`/`agg`.

In [ ]:
def task17_method_agg_orbital(planets: pd.DataFrame) -> pd.DataFrame:
    """min/median/max orbital_period по method."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 17 (не изменять)
out = task17_method_agg_orbital(planets)
assert list(out.columns) == ["min","median","max"]
assert out.index.is_unique
# sanity-check: min <= median <= max (где не NaN)
mask = out.notna().all(axis=1)
assert (out.loc[mask, "min"] <= out.loc[mask, "median"]).all()
assert (out.loc[mask, "median"] <= out.loc[mask, "max"]).all()

### GroupBy.filter: оставить методы с отношением max/min больше порога

**Задание 18.** Напиши `task18_filter_methods(planets, ratio=10000)`, которая:
- группирует по `method`;
- оставляет только те группы, где `max(orbital_period) / min(orbital_period) > ratio`;
- возвращает отфильтрованный DataFrame.

Важно: при вычислении `min`/`max` пропуски должны игнорироваться (стандартное поведение Pandas).

In [ ]:
def task18_filter_methods(planets: pd.DataFrame, ratio: float = 10000) -> pd.DataFrame:
    """Фильтрует группы method по условию max/min > ratio."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 18 (не изменять)
out = task18_filter_methods(planets, ratio=500)
assert isinstance(out, pd.DataFrame)
# Все оставшиеся методы должны удовлетворять условию
g = out.groupby("method")["orbital_period"]
cond = (g.max() / g.min()) > 500
assert cond.all()

### GroupBy.transform: центрирование orbital_period по методу

**Задание 19.** Напиши `task19_center_orbital_by_method(planets)`, которая:
- добавляет колонку `cntr_orbital_period`;
- `cntr_orbital_period = orbital_period - mean(orbital_period по method)`;
- использует `groupby(...).transform(...)` (как в лекции);
- возвращает копию DataFrame (не меняй входной объект).

In [ ]:
def task19_center_orbital_by_method(planets: pd.DataFrame) -> pd.DataFrame:
    """Добавляет cntr_orbital_period через groupby().transform()."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 19 (не изменять)
out = task19_center_orbital_by_method(planets)
assert "cntr_orbital_period" in out.columns
# Проверим, что среднее по группам близко к 0 (игнорируя NaN)
m = out.groupby("method")["cntr_orbital_period"].mean()
m = m.dropna()
assert (m.abs() < 1e-9).all()
# Входной объект не должен получить новую колонку
assert "cntr_orbital_period" not in planets.columns

### End-to-end: чистка + merge + groupby + pivot для розничных транзакций

**Задание 20.** Напиши `task20_retail_monthly_pivot(transactions, products, customers)`, которая строит итоговую таблицу выручки для дашборда.

Требования:
1) **Очистка:**
   - удалить строки, где `units <= 0`;
   - заполнить пропуски `price`: сначала `products['base_price']` по `product_id`, если всё ещё `NaN` — медианой по `price` (после первого заполнения);
   - `promo`: считать скидку 5% только если `promo == 1`, иначе скидка 0%.
2) **Расчёт:**
   - `revenue = units * price * (1 - 0.05 * (promo == 1))`
3) **Обогащение:**
   - присоединить `category` из `products` и `region` из `customers`.
4) **Агрегирование:**
   - добавить `month = date.astype('datetime64[ns]').dt.to_period('M')`;
   - посчитать сумму `revenue` по `month, region, category`.
5) **Форма результата:**
   - вернуть `DataFrame` с индексом `month` (PeriodIndex, freq='M');
   - колонки — `MultiIndex` (уровни: `region`, `category`);
   - значения — выручка.

**Важно:** решение должно быть векторизованным. В тестах есть проверка на скорость на увеличенных данных.

In [ ]:
def task20_retail_monthly_pivot(transactions: pd.DataFrame, products: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    """Строит monthly pivot выручки по region и category."""
    # YOUR CODE HERE

In [ ]:
# Тесты к заданию 20 (не изменять)
import time

out = task20_retail_monthly_pivot(transactions, products, customers)
assert isinstance(out, pd.DataFrame)
assert isinstance(out.index, pd.PeriodIndex)
assert out.index.freqstr == "M"
assert isinstance(out.columns, pd.MultiIndex)
assert list(out.columns.names) == ["region","category"]
assert (out.values >= 0).all()

# Проверка равенства суммарной выручки с «эталонным» groupby (без pivot)
# (Эталон реализован здесь намеренно коротко, чтобы не подсказывать структуру решения целиком.)
tx = transactions.copy()
tx = tx.loc[tx["units"] > 0].copy()
base_price = products.set_index("product_id")["base_price"]
tx["price"] = tx["price"].fillna(tx["product_id"].map(base_price))
tx["price"] = tx["price"].fillna(tx["price"].median())
disc = (tx["promo"] == 1).astype(float)
tx["revenue"] = tx["units"] * tx["price"] * (1 - 0.05*disc)
tx = tx.merge(products[["product_id","category"]], on="product_id", how="left")
tx = tx.merge(customers[["customer_id","region"]], on="customer_id", how="left")
tx["month"] = pd.to_datetime(tx["date"]).dt.to_period("M")
ref = tx.groupby(["month","region","category"])["revenue"].sum()

assert np.isclose(out.stack([0,1]).sort_index().values.sum(), ref.sum())

# Тест на скорость: увеличим транзакции (циклы будут «умирать»)
big = pd.concat([transactions]*25, ignore_index=True)
t0 = time.perf_counter()
_ = task20_retail_monthly_pivot(big, products, customers)
dt = time.perf_counter() - t0
assert dt < 2.5, f"Слишком медленно ({dt:.2f}s). Вероятно, использованы циклы или построчные операции."


## Источники
1. Макрушин С. В. *Лекция 2: Библиотека Pandas*.  
   (В том числе: Series и индексация; DataFrame как «словарь Series»; `loc/iloc`; маскирование; выравнивание; NaN; агрегирование; merge; groupby: split-apply-combine.)
2. McKinney W. *Python for Data Analysis* (O'Reilly).
